# Smoke de conectividad HTTP externa

Comprueba que hay **salida a Internet** hacia los hosts que usa el código: Binance (spot, futures, Vision), **CryptoCompare** (fallback de huecos OHLCV), **Google Trends** (frontal HTTPS; los datos van vía `trendspy`); un **429** ahí cuenta como OK (*rate limit*, red alcanzable), **The Guardian** y **The New York Times**, **PullPush** y **Arctic Shift** (Reddit), **REST de Wikipedia** (`page/summary` en `en.wikipedia.org`; distinto del host de métricas pageviews en `wikimedia.org`) y **Dune** (on-chain).

**Importante:** no valida claves de API ni cuotas. Respuestas **401/403/404/429** pueden contar como OK si demuestran que hubo respuesta HTTP del servicio. No sustituye al gate reproducible offline del CI.

**Cuándo usarlo:** evidencia manual de red y disponibilidad de terceros; tabla por categoría con latencia aproximada.

**Requisito:** `RUN_EXTERNAL_SMOKE=1` (la celda de configuración lo activa) y equipo con HTTPS saliente.

**Otros servicios del repo no incluidos aquí:** tracking **MLflow** si apunta a un servidor remoto (configura URI en tu entorno), modelos **Hugging Face** (FinBERT) en la primera descarga, y cualquier **TimescaleDB**, ya que es conectividad de red interna de Docker y no HTTP público en esta tabla.

## Uso recomendado

**Prerrequisitos:** completar `00_infraestructura/01_smoke_test_infraestructura.ipynb` y, si aplica, `00_infraestructura/02_evidencia_infraestructura.ipynb` (stack Docker/BD). Este notebook es **opcional** y no bloquea el pipeline local.

**Orden sugerido del pilar:** `01` (smoke interno) -> `02` (evidencia contractual) -> `03` (este notebook, red externa).

**Paridad con tests:** misma lógica que `tests/integration/test_external_connectivity_smoke.py` (`pytest -m external_smoke` con `RUN_EXTERNAL_SMOKE=1`).

**Ejecución:** `Kernel > Restart Kernel and Run All` tras activar `RUN_EXTERNAL_SMOKE` en la celda de configuración.

## Artefactos bajo `reports/`

No se exportan figuras ni CSV/JSON a `reports/`. La salida es la tabla en pantalla y el `assert` final. Para un gate reproducible en terminal, usar pytest (no este cuaderno como sustituto del CI).

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

# Localiza la raíz del repo validando `src/` y `tests/integration/`
ROOT = next(
    (
        p
        for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
        if (p / "src").is_dir() and (p / "tests" / "integration").is_dir()
    ),
    None,
)
if ROOT is None:
    raise RuntimeError("No se encontró la raíz del repo (falta src/ o tests/integration/).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tests.integration.test_external_connectivity_smoke import (
    run_external_smoke_checks,
    external_smoke_enabled,
)

os.environ["RUN_EXTERNAL_SMOKE"] = "1"
print("RUN_EXTERNAL_SMOKE activado para esta sesión del cuaderno")
print("external_smoke_enabled() ->", external_smoke_enabled())

RUN_EXTERNAL_SMOKE activado para esta sesión del cuaderno
external_smoke_enabled() -> True


In [2]:
import pandas as pd

# Construye tabla de resultados normalizada para lectura rápida
df = pd.DataFrame(
    [
        {
            "categoría": result.category,
            "endpoint": result.name,
            "ok": result.ok,
            "latencia_ms": result.latency_ms,
            "detalle": result.detail,
        }
        for result in run_external_smoke_checks()
    ]
)

# Aplica control de fallo explícito cuando exista algún endpoint no alcanzable
display(df)
assert df["ok"].all(), (
    "Una o más comprobaciones fallaron; revisa firewall, DNS o bloqueos regionales. "
    "Detalle: "
    + "; ".join(df.loc[~df["ok"], "detalle"].astype(str).tolist())
)

,categoría,endpoint,ok,latencia_ms,detalle
0,Binance spot REST,api.binance.com — GET /api/v3/ping,True,5019.60,http_status=200
1,Binance Futures REST (UM),fapi.binance.com — GET /fapi/v1/ping,True,1023.51,http_status=200
2,Binance Vision (datasets HTTPS),data.binance.vision — GET raíz data.binance.vi...,True,5462.09,http_status=200
3,OHLCV gap / fallback horario,min-api.cryptocompare.com — GET min-api.crypto...,True,2672.39,http_status=200
4,Google Trends (frontal HTTPS),trends.google.com — GET trends.google.com,True,1429.66,http_status=429 (alcance OK; 429 suele ser rat...
5,The Guardian Open Platform,content.guardianapis.com — GET content.guardia...,True,2409.99,http_status=401 (solo conectividad; respuesta ...
6,The New York Times API,api.nytimes.com — GET api.nytimes.com (esperab...,True,1009.72,http_status=401 (solo conectividad; respuesta ...
7,Reddit Pullpush,api.pullpush.io — GET api.pullpush.io/reddit/s...,True,15442.28,http_status=200
8,Reddit Arctic Shift,arctic-shift.photon-reddit.com — GET arctic-sh...,True,1648.67,http_status=200
9,Wikipedia / Wikimedia REST,en.wikipedia.org — GET page/summary en en.wiki...,True,1308.96,http_status=200
